In [ ]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cuda", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        if damping == "zero":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 1.261),
                        kwargs.get("s18", 1.703),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": kwargs.get("rs18", 1.0),
                "alp": kwargs.get("alp", 14.0),
            }
        elif damping == "bj":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 0.3981),
                        kwargs.get("s18", 1.9889),
                        kwargs.get("rs18", 4.4211),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": self.param_vector[2],
                "alp": kwargs.get("alp", 14.0),
            }
        self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, batch_dicts):
        self.calc.reset()

        # Calculate the energy using the DFTD3 calculator
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal

    def obtain_batch_dicts(self, atoms_list):
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        return batch_dicts


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1513512_gmtkn-cc-pVDZ.csv"
)
data_name_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
data_cc_ene = data["cc_ene"].to_numpy() * 627.5094733748099
data_dft_ene = data["scf_ene"].to_numpy() * 627.5094733748099
batch_subset = [
    "W4_11",
    # "G21EA",
    # "G21IP",
    # "DIPCS10",
    # "PA26",
    # "SIE4x4",
    # "ALKBDE10",
    # "YBDE18",
    # "AL2X6",
    # "HEAVYSB11",
    # "NBPRC",
    # "ALK8",
    # "RC21",
    # "G2RC",
    # "BH76RC",
    # "FH51",
    # "TAUT15",
    # "DC13",
    # "MB16_43",
    # "DARC",
    # "RSE43",
    # "BSR36",
    # "CDIE20",
    # "ISO34",
    # # "ISOL24",
    # # "C60ISO",
    # "PArel",
    # "BH76",
    # "BHPERI",
    # "BHDIV10",
    # "INV24",
    # "BHROT27",
    # "PX13",
    # "WCPT18",
    # "RG18",
    # "ADIM6",
    # "S22",
    # "S66",
    # # "HEAVY28",
    # "WATER27",
    # "CARBHB12",
    # "PNICO23",
    # "HAL59",
    # "AHB21",
    # "CHB6",
    # "IL16",
    # "IDISP",
    # "ICONF",
    # "ACONF",
    # "Amino20x4",
    # "PCONF21",
    # "MCONF",
    # "SCONF",
    # # "UPU23",
    # "BUT14DIOL",
]

with open(f"new_dataset/gmtkn-cc-pVDZ.json") as f:
    json_data = json.load(f)

input_batch = {}
name_batch_list = {}
weight_batch_list = {}
mean_absolute_deviation = []
model = Model(device="cuda", damping="bj")
# model = Model(device="cuda", damping="zero")
model.compile(mode="max-autotune-no-cudagraphs")
for name_mol in data_name_list:
    for i_subset in batch_subset:
        if i_subset == "BH76RC":
            i_subset_name = "BH76"
        else:
            i_subset_name = i_subset
        if name_mol.startswith(i_subset_name):
            mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
            atoms = Atoms(
                symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
            )
            if i_subset not in input_batch:
                input_batch[i_subset] = []
            input_batch[i_subset].append(atoms)
            if i_subset not in name_batch_list:
                name_batch_list[i_subset] = []
            name_batch_list[i_subset].append(name_mol)

for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]
    name_batch_list[i_subset] = np.array(name_batch_list[i_subset])
    input_batch[i_subset] = model.obtain_batch_dicts(input_batch[i_subset])
    reaction_dict_copy = reaction_dict.copy()
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict_copy.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            if col.size != 1:
                print(f"Warning: {mole_name} not found in name_list")
                reaction_dict.pop(i_reaction_keys)
                break
    json_data[f"reaction-{i_subset}"] = reaction_dict

energy_batch_target = {}
for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]

    energy_batch_target[i_subset] = torch.zeros(
        len(reaction_dict), dtype=torch.float64
    )
    weight_batch = np.zeros(len(reaction_dict), dtype=np.float64)
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]
        energy_dft = 0

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            energy_dft += (data_cc_ene[col[0]] - data_dft_ene[col[0]]) * stoichiometry
            weight_batch[i_reaction_name] += data_cc_ene[col[0]] * stoichiometry
        energy_batch_target[i_subset][i_reaction_name] = energy_dft
    mean_absolute_deviation.extend(np.abs(weight_batch))
    weight_batch_list[i_subset] = 1 / np.mean(np.abs(weight_batch))

print(
    f"mean_absolute_deviation: {np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
loss_function = torch.nn.L1Loss(reduction="sum")
torch.set_printoptions(precision=5, sci_mode=False)
energy_batch_output = {}
print("start training...")

def printable(epoch):
    if epoch % 100 == 0:
        return True
    return False
if_print_step = True

for epoch in tqdm.tqdm(range(2501)):
    loss_batch = []
    wtmad_2 = 0
    optimizer.zero_grad()
    for i_subset in batch_subset:
        energy = model(input_batch[i_subset])

        reaction_dict = json_data[f"reaction-{i_subset}"]
        energy_batch_output[i_subset] = torch.zeros(
            len(reaction_dict), dtype=torch.float64
        )
        for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
            reaction_dict.items()
        ):
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]
            energy_dft = 0

            for i in range(len(systems_list)):
                mole_name = (
                    systems_list[i]
                    if i_subset == "BH76RC"
                    else f"{i_subset}-{systems_list[i]}"
                )
                stoichiometry = int(stoichiometry_list[i])

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]

                col_disp = np.where(name_batch_list[i_subset] == mole_name)[0]
                if col_disp.size == 1:
                    energy_dft += energy[col_disp[0]] * stoichiometry
                else:
                    print(f"Warning: {mole_name} not found in name_list")
                    break
            energy_batch_output[i_subset][i_reaction_name] = energy_dft
        loss = (
            loss_function(energy_batch_output[i_subset], energy_batch_target[i_subset])
            * weight_batch_list[i_subset]
        )
        loss_batch.append(
            torch.mean(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            ).item()
        )
        if printable(epoch) and if_print_step:
            print(
                f"{i_subset}, params: {model.calc.dftd_module.params}, mean(|E|): {torch.mean(torch.abs(energy_batch_target[i_subset])).item()}, EACH: {energy_batch_output[i_subset] - energy_batch_target[i_subset]}"
            )
        wtmad_2 += (
            torch.sum(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            )
            * weight_batch_list[i_subset]
        ).item()
        # clip the loss to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        loss.backward()
    optimizer.step()

    if printable(epoch):
        print(
            f"Epoch: {epoch}, wtmad_2: {wtmad_2 * np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}, loss: {loss_batch}"
        )

print(f"params_vector {model.calc.dftd_module.params}")

mean_absolute_deviation: 1.907452919650298
start training...


  0%|          | 0/2501 [00:00<?, ?it/s]

W4_11, params: {'s6': 1.0, 'rs6': tensor(0.39810, device='cuda:0', dtype=torch.float64,
       grad_fn=<SelectBackward0>), 's18': tensor(1.98890, device='cuda:0', dtype=torch.float64,
       grad_fn=<SelectBackward0>), 'rs18': tensor(4.42110, device='cuda:0', dtype=torch.float64,
       grad_fn=<SelectBackward0>), 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.30013,      0.53794,      1.03246,      1.29847,      0.37621,
            -0.28355,      0.06165,      1.89525,     10.35723,      0.28638,
             0.24946,     -4.47341,      9.99628,      0.75359,      3.16423,
             6.01381,     -0.22225,      1.50342,      0.32446,      0.98313,
            10.89390,      3.95256,      7.26138,      5.91581,     -0.04297,
             8.25969,      5.68296,      1.94281,      5.14819,     -0.21476,
             0.54850,     -0.14887,      6.84193,     16.70592,     -0.03127,
             0.25215,      3.88717,      0.37047,      7.61492,      8.83306,
          

  0%|          | 2/2501 [00:52<15:06:45, 21.77s/it]

Epoch: 0, wtmad_2: 10.371244070447306, loss: [10.371244070447306]


...

 52%|█████▏    | 1301/2501 [03:51<02:39,  7.54it/s]

W4_11, params: {'s6': 1.0, 'rs6': tensor(0.50514, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 's18': tensor(1.88072, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'rs18': tensor(4.52856, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.33798,     -0.09356,      0.73501,      0.32181,     -0.03756,
            -0.44069,     -0.18241,      1.63204,      7.45579,     -0.11272,
            -0.29371,     -6.20504,      8.94853,      0.01364,      1.64272,
             3.27549,     -0.46807,      1.37711,     -0.10902,      0.77989,
             9.27311,      2.72711,      6.65293,      3.63977,     -0.38600,
             6.30588,      4.65721,      0.79635,      4.26526,     -0.38840,
             0.33644,     -0.24230,      5.80686,     16.17194,     -0.19271,
             0.20172,      2.97468,      0.30030,      5.97205,      7.28558,
 

 56%|█████▌    | 1401/2501 [04:04<02:38,  6.94it/s]

W4_11, params: {'s6': 1.0, 'rs6': tensor(0.51202, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 's18': tensor(1.87369, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'rs18': tensor(4.53549, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.33968,     -0.11918,      0.72330,      0.28125,     -0.05526,
            -0.44726,     -0.19292,      1.62147,      7.33588,     -0.13003,
            -0.31744,     -6.27958,      8.90520,     -0.01712,      1.57571,
             3.15351,     -0.47864,      1.37173,     -0.12703,      0.77152,
             9.20189,      2.67319,      6.62650,      3.53992,     -0.40107,
             6.21952,      4.61251,      0.74689,      4.22663,     -0.39562,
             0.32718,     -0.24635,      5.76173,     16.14938,     -0.19980,
             0.19950,      2.93570,      0.29724,      5.90081,      7.21819,
 

 59%|█████▉    | 1486/2501 [04:16<02:25,  6.97it/s]

In [ ]:
data_dft_bj = []
model.calc.dftd_module.params = 

for name_mol in data_name_list:
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(symbols=mol.elements, positions=mol.atom_coords() * units.Bohr)
    energy = model(model.obtain_batch_dicts([atoms]))
    print(f"{name_mol}: {energy.item():.10f} kcal/mol")
    data_dft_bj.append(energy.item() / 627.5094733748099)

data["modified_dft_d3zero"] = data_dft_bj
data.to_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-2453600_gmtkn-cc-pVDZ.csv",
    index=False,
)

BSR36-ch4: -0.2174092661 kcal/mol
BSR36-c2h6: -2.3199845812 kcal/mol
BSR36-r1: -10.6162756838 kcal/mol
BSR36-h1: -18.3407325456 kcal/mol
BSR36-c4: -53.0955226743 kcal/mol


# modified_ai_d3zero
params_vector {'s6': 1.0, 'rs6': tensor(1.5055195208, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 's18': tensor(1.4496851945, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'rs18': 1.0, 'alp': 14.0}